In [0]:
from pyspark.sql import functions as F

# ============================================================
# 06_MARK_SUCCESS
# Enregistre le feed comme traité avec succès
# ============================================================

CONTROL_TABLE = "workspace.sncf_bronze.pipeline_control"

# Récupérer les valeurs produites par la task pipeline_control
feed_start = dbutils.jobs.taskValues.get(
    taskKey="pipeline_control",
    key="feed_start_date"
)

feed_end = dbutils.jobs.taskValues.get(
    taskKey="pipeline_control",
    key="feed_end_date"
)

print("Feed traité :", feed_start, "->", feed_end)

# ============================================================
# Création de l'enregistrement SUCCESS
# ============================================================

df_success = (
    spark.createDataFrame(
        [(feed_start, feed_end, "SUCCESS")],
        ["feed_start_date", "feed_end_date", "status"]
    )
    .withColumn(
        "feed_start_date",
        F.to_date("feed_start_date")
    )
    .withColumn(
        "feed_end_date",
        F.to_date("feed_end_date")
    )
    .withColumn(
        "processed_at",
        F.current_timestamp()
    )
    .select(
        "feed_start_date",
        "feed_end_date",
        "processed_at",
        "status"
    )
)

# ============================================================
# Évite d'insérer un doublon SUCCESS
# ============================================================

already_exists = (
    spark.table(CONTROL_TABLE)
    .filter(
        (F.col("feed_start_date") == F.to_date(F.lit(feed_start))) &
        (F.col("feed_end_date") == F.to_date(F.lit(feed_end))) &
        (F.col("status") == "SUCCESS")
    )
    .limit(1)
    .count() > 0
)

if already_exists:
    print("Ce feed est déjà enregistré comme SUCCESS.")

else:
    (
        df_success.write
        .format("delta")
        .mode("append")
        .saveAsTable(CONTROL_TABLE)
    )

    print("Feed enregistré avec succès dans pipeline_control.")

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-5945394562645323>, line 11
      8 CONTROL_TABLE = "workspace.sncf_bronze.pipeline_control"
     10 # Récupérer les valeurs produites par la task pipeline_control
---> 11 feed_start = dbutils.jobs.taskValues.get(
     12     taskKey="pipeline_control",
     13     key="feed_start_date"
     14 )
     16 feed_end = dbutils.jobs.taskValues.get(
     17     taskKey="pipeline_control",
     18     key="feed_end_date"
     19 )
     21 print("Feed traité :", feed_start, "->", feed_end)

File /databricks/python_shell/lib/dbruntime/dbutils.py:288, in DBUtils.JobsHandler.TaskValuesHandler.get(self, taskKey, key, default, debugValue)
    286 elif is_not_in_job_context:
    287     if debugValue is None:
--> 288         raise patch_exception_with_error_details(
    289             TypeError(
    290                 'Must pass debugV